In [2]:
!pip install PineCone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [PineCone]1/2 [PineCone]


In [15]:
from pinecone import Pinecone, PodSpec, ServerlessSpec
from sentence_transformers import SentenceTransformer

In [16]:
model_name = 'distilbert-base-nli-stsb-mean-tokens'
model = SentenceTransformer(model_name)

In [ ]:
pinecone_key = 'p'
pc = Pinecone(api_key=pinecone_key)

In [18]:
pc.list_indexes()

[]

In [19]:
pc.create_index(
    name = 'vector-demo',
    dimension=768,
    metric='euclidean',
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "vector-demo",
    "metric": "euclidean",
    "host": "vector-demo-j4pby3t.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2

In [20]:
data = [
    {"id": "vector1",  "text": "I love using vector databases"},
    {"id": "vector2",  "text": "Vector databases are great for storing and retrieving vectors"},
    {"id": "vector3",  "text": "Using vector databases makes my life easier"},
    {"id": "vector4",  "text": "Vector databases are efficient for storing vectors"},
    {"id": "vector5",  "text": "I enjoy working with vector databases"},
    {"id": "vector6",  "text": "Vector databases are useful for many applications"},
    {"id": "vector7",  "text": "I find vector databases very helpful"},
    {"id": "vector8",  "text": "Vector databases can handle large amounts of data"},
    {"id": "vector9",  "text": "I think vector databases are the future of data storage"},
    {"id": "vector10", "text": "Using vector databases has improved my workflow"}
]

In [21]:
vector_data = []
for sentences in data:
    embedding = model.encode(sentences["text"])
    vector_info = {'id':sentences['id'], 'values':embedding.tolist()}
    vector_data.append(vector_info)
    

In [24]:
index = pc.Index('vector-demo')

In [25]:
index.upsert(vectors=vector_data)

UpsertResponse(upserted_count=10, _response_info={'raw_headers': {'date': 'Sat, 28 Mar 2026 20:51:18 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '30791', 'x-pinecone-request-latency-ms': '295', 'x-envoy-upstream-service-time': '243', 'x-pinecone-response-duration-ms': '297', 'grpc-status': '0', 'server': 'envoy'}})

In [26]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '186',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 28 Mar 2026 20:52:36 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '3',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'euclidean',
 'namespaces': {'__default__': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}

In [27]:
search_text = "Vector Databases are really helpful"
search_embedding = model.encode(search_text).tolist()

In [28]:
index.query(vector=search_embedding, top_k=3)

QueryResponse(matches=[{'id': 'vector7', 'score': 21.0021057, 'values': []}, {'id': 'vector4', 'score': 44.0501404, 'values': []}, {'id': 'vector5', 'score': 46.4408569, 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sat, 28 Mar 2026 20:54:41 GMT', 'content-type': 'application/json', 'content-length': '209', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '5', 'x-envoy-upstream-service-time': '5', 'x-pinecone-response-duration-ms': '7', 'grpc-status': '0', 'server': 'envoy'}})